# 基于paddle的高层API实现简单线性回归模型
“机器学习概述-上.ipynb” 中的简单线性回归模型基本采用自定义方式实现，没有调用paddle的高层API。本任务仍然使用任务一中的定义的数据集，采用paddle的高层API来实现线性回归。

In [ ]:
import datetime
import os

def printLog(name, ID):
    # 打印和
    print(': %s; : %s' % (name, ID))
    # 打印操作系统的用户名
    print("系统用户名:", os.getlogin())
    # 打印当前时间
    print('运行时间:', datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

"""*********************************************************************
   【注意】运行前，请把下面代码中的“张三”改为自己的，123456改为自己的。
   *********************************************************************"""
printLog('', )

# 0. 相关类说明
## 0.1 paddle.nn.Linear（用于模型定义）
__`class paddle.nn.Linear(in_features, out_features, weight_attr=None, bias_attr=None, name=None)`__
`paddle.nn.Linear` 用于实现全连接层（也称为线性层或密集层）。全连接层是神经网络中最基本的层之一，它的功能是将输入数据进行线性变换，通常用于网络的最后几层，以输出最终的预测结果。</br>
主要参数：
* in_features：输入特征的数量，即输入数据的维度。
* out_features：输出特征的数量，即输出数据的维度。
* bias：一个布尔值，表示是否使用偏置项。默认为 True，即使用偏置项。

__<span style="color: red;">paddle.nn.Linear官方文档请参考：</span>__ 
https://www.paddlepaddle.org.cn/documentation/docs/zh/api/paddle/nn/Linear_cn.html#cn-api-paddle-nn-linear

下面是一个使用paddle.nn.Linear的简单示例，演示了如何在一个简单的神经网络中使用它。

In [ ]:
import paddle  
from paddle import nn  
  
# 定义一个简单的网络  
class SimpleNet(nn.Layer):  
    def __init__(self):  
        super(SimpleNet, self).__init__()  
        # 定义一个全连接层，输入特征数为10，输出特征数为5  
        self.fc = nn.Linear(in_features=10, out_features=5)  
  
    def forward(self, x):  
        # 通过全连接层  
        x = self.fc(x)  
        return x  

# 实例化模型  
model = SimpleNet()  
  
# 创建一个随机输入数据，形状为[batch_size, in_features]  
# 这里batch_size设为1，in_features为10  
input_data = paddle.randn([1, 10], dtype='float32')  
print("输入数据:", input_data)
  
# 前向传播  
output = model(input_data)  
  
print("输出数据:", output)

## 0.2 paddle.nn.MSELoss（损失函数）
__`paddle.nn.MSELoss(reduction='mean')`__</br>
`paddle.nn.MSELoss()` 是 PaddlePaddle 框架中用于计算均方误差（Mean Squared Error, MSE）损失的一个类。MSE 损失是回归任务中最常用的损失函数之一，它衡量的是真实值与预测值之间差异的平方和的平均值。</br>
主要参数：
* reduction (str, 可选)：指定损失输出的缩减方式，可以是 'none', 'mean', 或者 'sum'。默认为 'mean'，表示将所有样本的损失求平均；如果设置为 'sum'，则输出所有样本的损失之和；如果设置为 'none'，则输出每个样本的损失值。

__<span style="color: red;">paddle.nn.MSELoss官方文档请参考：</span>__ 
https://www.paddlepaddle.org.cn/documentation/docs/zh/api/paddle/nn/MSELoss_cn.html#mseloss

下面是一个简单的例子来展示如何使用 paddle.nn.MSELoss：

In [ ]:
import paddle
from paddle.nn import MSELoss

# 创建一个 MSE Loss 计算对象
criterion = MSELoss()

# 假设我们有两个形状相同的张量作为输入和目标
input_tensor = paddle.to_tensor([[1.0, 2.0], [3.0, 4.0]], dtype='float32')
target_tensor = paddle.to_tensor([[0.5, 2.0], [3.0, 4.0]], dtype='float32')

# 计算 MSE Loss
loss = criterion(input_tensor, target_tensor)

print("MSE Loss:", loss.numpy())  # 输出损失值

## 0.3 paddle.optimizer.Adam（优化器）
__`paddle.optimizer.Adam(parameters, learning_rate=0.001, beta1=0.9, beta2=0.999, epsilon=1e-08, weight_decay=None, grad_clip=None, name=None, lazy_mode=False)`__</br>
`paddle.optimizer.Adam` 是 PaddlePaddle中的一个优化器类，用于实现 Adam 优化算法。Adam（Adaptive Moment Estimation）是一种自适应学习率的方法，它结合了动量（Momentum）和RMSProp的概念，用于深度学习中的模型参数优化。</br>
主要参数：
* parameters (list|tuple) - 待优化参数的列表或元组，通常是模型参数的迭代器。
* learning_rate (float, 可选) - 初始学习率，默认为 0.001。

__<span style="color: red;">paddle.optimizer.Adam官方文档请参考：</span>__ https://www.paddlepaddle.org.cn/documentation/docs/zh/api/paddle/optimizer/Adam_cn.html#adam

以下是一个使用 paddle.optimizer.Adam 的简单示例，包括定义一个简单的线性模型、定义 Adam 优化器、以及执行一次前向传播和反向传播来更新模型参数的过程。

In [ ]:
import paddle  
from paddle import nn  
  
# 定义一个简单的线性模型  
class LinearModel(nn.Layer):  
    def __init__(self):  
        super(LinearModel, self).__init__()  
        self.linear = nn.Linear(in_features=10, out_features=1)  
  
    def forward(self, x):  
        return self.linear(x)  
  
# 实例化模型  
model = LinearModel()  
  
# 假设的输入和标签  
inputs = paddle.randn([32, 10], dtype='float32')  
labels = paddle.randn([32, 1], dtype='float32')  
  
# 定义损失函数  
criterion = nn.MSELoss()  
# 定义 Adam 优化器，指定要优化的参数为模型的所有参数  
optimizer = paddle.optimizer.Adam(parameters=model.parameters())  
# 前向传播  
preds = model(inputs)  
# 计算损失  
loss = criterion(preds, labels)  
# 反向传播  
loss.backward()  
# 更新参数  
optimizer.step()  
# 清除梯度，为下一次迭代做准备  
optimizer.clear_grad()  
  
print("Loss:", loss.numpy())

# 1. 数据集构建

In [ ]:
import paddle
import paddle.nn as nn
import paddle.optimizer as optim
import numpy as np
from matplotlib import pyplot as plt

In [ ]:
# 采用“HW2-1.机器学习概述-上.ipynb”中的数据集构建方法创建训练数据集X_train, y_train和测试数据集X_test, y_test，并绘制数据
# 即采用linear_func、create_toy_data函数实现
# 请将数据的分布修改为 y = 1.5*x + 0.6，训练样本数修改为500，测试样本数修改为200，其他参数设置保持不变
# 生成训练数据和测试数据
def linear_func(x):
    return 1.5 * x + 0.6

def create_toy_data(func, sample_size, std=0.2, random_seed=42):
    np.random.seed(random_seed)
    x = np.random.uniform(0, 1, sample_size)
    y = func(x) + np.random.normal(scale=std, size=sample_size)
    return x, y

X_train, y_train = create_toy_data(linear_func, 500)
X_test, y_test = create_toy_data(linear_func, 200)

In [ ]:
"执行下面的基于Paddle API实现的模型定义和训练前，要按照下面的方法重新修改训练数据X_train、y_train的shape，为什么？"
x = X_train.reshape([-1,1])
y = y_train.reshape([-1,1])
print('x:',x)
print('y:',y)

# 2. 基于Paddle高层API的线性回归模型定义

In [ ]:
# 使用paddle.nn.Linear定义线性回归模型LinearRegression()
class LinearRegression(nn.Layer):
    def __init__(self):
        super(LinearRegression, self).__init__()
        self.linear = nn.Linear(1, 1)
    
    def forward(self, x):
        return self.linear(x)

# 将numpy数组转换为Paddle Tensor
x = paddle.to_tensor(x, dtype='float32')
y = paddle.to_tensor(y, dtype='float32')

# 3. 基于Paddle高层API的模型训练

In [ ]:
# 实例化模型、损失函数和优化器
model = LinearRegression()  #  实例化模型
loss_fn = nn.MSELoss()  #  均方损失函数
optimizer = optim.Adam(parameters=model.parameters(), learning_rate=0.1)  #  Adam优化器，是对动量梯度下降法的改进

# 训练模型
epochs = 200
for epoch in range(epochs):
    # 前向传播
    pred = model(x)
    loss = loss_fn(pred, y)
    
    # 反向传播和优化
    optimizer.clear_grad()  # 清除梯度
    loss.backward()     # 反向传播计算梯度
    optimizer.step()    # 更新参数
    
    # 打印损失
    if (epoch+1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.numpy()}")

# 单个数据的预测
x_new = np.array([[5.0]], dtype='float32')  # 给定一个值5.0用于测试训练后的模型，真实模型为 y = 1.5*x + 0.6
x_new_paddle = paddle.to_tensor(x_new, dtype='float32')
pred_new = model(x_new_paddle)  # 用模型预测新值
print(f"Predicted value: {pred_new.numpy()}")  # 输出模型预测值（真实值应为 1.5*5.0 + 0.6 = 8.1）

# 保存训练后的模型
paddle.save(model.state_dict(), 'linear_regression_model.pdparams')

In [ ]:
# 打印训练后的模型参数
print("Model Parameters:")
for name, param in model.named_parameters():
    print(f"{name}: {param.numpy()}")
    if name == 'linear.weight':  # 权值w
        w = list(param.numpy())
    if name == 'linear.bias':    # 偏置b
        b = list(param.numpy())

In [ ]:
"""【注意】运行前，请把下面代码中的“张三”改为自己的，123456改为自己的。"""
printLog('', )

# 4. 模型评估
##  损失函数
回归任务是对**连续值**的预测，希望模型能根据数据的特征输出一个连续值作为预测值。因此回归任务中常用的评估指标是**均方误差**。

令$\boldsymbol{y}\in \mathbb{R}^N$，$\hat{\boldsymbol{y}}\in \mathbb{R}^N$分别为$N$个样本的真实标签和预测标签，均方误差的定义为：

$$
\mathcal{L}(\boldsymbol{y},\hat{\boldsymbol{y}})=\frac{1}{2N}\|\boldsymbol{y}-\hat{\boldsymbol{y}}\|^2=\frac{1}{2N}\|\boldsymbol{X}\boldsymbol{w}+\boldsymbol{b}-\boldsymbol{y}\|^2, (2.8)
$$
其中$\boldsymbol{b}$为$N$维向量，所有元素取值都为$b$。

均方误差的代码实现如下:
> 注意：代码实现中没有除2。

In [ ]:
# 采用“HW2-1.机器学习概述-上.ipynb”中的均方误差函数mean_squared_error对训练好的模型在测试数据集上测试数据集X_test, y_test进行评估
# 或者采用paddle的nn.MSELoss()函数对训练好的模型在测试数据集上测试数据集X_test, y_test进行评估
# 打印输出测试集上的平均误差
X_test_reshaped = X_test.reshape([-1, 1])
y_test_reshaped = y_test.reshape([-1, 1])
X_test_tensor = paddle.to_tensor(X_test_reshaped, dtype='float32')
y_test_tensor = paddle.to_tensor(y_test_reshaped, dtype='float32')

# 使用训练好的模型进行预测
model.eval()
with paddle.no_grad():
    test_pred = model(X_test_tensor)
    test_loss = loss_fn(test_pred, y_test_tensor)

print(f"测试集上的均方误差: {test_loss.numpy()}")

# 也可以使用自定义的均方误差函数
def mean_squared_error(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

# 使用numpy计算
test_pred_np = test_pred.numpy()
mse_numpy = mean_squared_error(y_test_reshaped, test_pred_np)
print(f"使用numpy计算的均方误差: {mse_numpy}")


In [ ]:
"""【注意】运行前，请把下面代码中的“张三”改为自己的，123456改为自己的。"""
printLog('', )